[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/TP_reseau_energie.ipynb)

# Le réseau d'énergie : architecture, scores, débruitage et normalisation

On ouvre le réseau d'énergie entraîné par les auteurs sur ImageNet64 : $U_\theta(y,t) = \frac12\langle y, s_\theta(y,t)\rangle$, où $s_\theta$ est un UNet. On vérifie ce que l'article affirme de cette architecture (gradient, homogénéité), on mesure ses performances de débruitage (tableau 1), on contrôle la cohérence de ses deux scores (équation de diffusion) et sa normalisation (équation (9)).

**Matériel** : tourne sur un ordinateur personnel (CPU) en quelques minutes ; sur Colab, exécuter d'abord la cellule d'installation.

**Profils** : sur CPU, les mesures portent sur quelques dizaines d'images (environ 2 minutes en tout) ; sur GPU, sur quelques centaines. Le notebook des auteurs correspondant au tableau 1 est `auteurs/denoiser_vs_energy_colab.ipynb`.

**Reprise** : les calculs longs sont mis en cache dans `resultats/cache/` ; une exécution interrompue reprend là où elle s'était arrêtée. Supprimer ce dossier pour tout recalculer.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py',
            '/models/finalclean_imagenet64_grayscale_lr0.0005_1Msteps10decays/',
            '/models/finalclean_imagenet64_color_lr0.0005_1Msteps10decays/',
            '/models/finalclean_imagenet64_color_denoiser_psco_lr0.0005_1Msteps10decays/']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, "tp"))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, "tp"))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

## 0. Réglages et données

Les images sont celles de la **validation** d'ImageNet64 (version de Chrabaszcz et al., celle des auteurs),
téléchargée au premier appel depuis un miroir HuggingFace (417 Mo). Aucun des réseaux ne les a vues pendant
l'entraînement.

In [ ]:
%matplotlib inline
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_reseau_energie")

GPU = torch.cuda.is_available()
N_IMAGES = 256 if GPU else 32          # images pour les mesures sur le réseau
N_HUTCHINSON = 16 if GPU else 4         # images pour l'équation de diffusion
GRAINE = 0
T.fixer_graine(GRAINE)
print("calcul sur", T.DEVICE)

m = T.charger_modele("gris")
d = m.d
X = T.charger_imagenet64_val(N_IMAGES)            # (N, 1, 64, 64) dans [0, 1]
T.afficher_images(X[:10], titre="images de validation (niveaux de gris)")

## 1. L'architecture

`Reparameterization` (fichier `networks/model.py`) enveloppe le UNet et fabrique le scalaire ; `EnergyModel` dérive
ce scalaire par autograd. On affiche la formule calculée, le nombre de paramètres, la présence de biais et les
fréquences de l'encodage du niveau de bruit $t$.

In [ ]:
print("formule :", m.reseau.network.extra_repr())
print("arguments du UNet :", m.args.network_kwargs, "; reparamétrisation :", m.args.reparam_kwargs)
print("paramètres : %.2f millions" % (m.nb_parametres / 1e6))
convs = [mod for mod in m.unet.modules() if isinstance(mod, (torch.nn.Conv2d, torch.nn.ConvTranspose2d))]
print("couches de convolution : %d, dont avec biais : %d" % (len(convs), sum(c.bias is not None for c in convs)))
print("largeurs (canaux de sortie) :", [c.out_channels for c in convs])
print("non-linéarités :", {type(mod).__name__ for mod in m.unet.modules() if isinstance(mod, (torch.nn.GELU, torch.nn.ReLU, torch.nn.SiLU))})
emb = m.unet.noise_var_embedding
print("encodage de t :", emb.extra_repr(), " (t_min, t_max de l'entraînement : %.0e, %.0e)" % (T.T_MIN, T.T_MAX))

## 2. L'énergie est un produit scalaire : gradient et sortie du UNet

$U_\theta(y,t) = \frac12\langle y, s_\theta(y,t)\rangle$ donne, sans hypothèse,
$\nabla_y U_\theta = \frac12\big(s_\theta + \nabla_y s_\theta^\top y\big)$ ; si $s_\theta$ est homogène et
conservatif, $\nabla_y U_\theta = s_\theta$ (annexe B.2). On mesure l'écart relatif
$\|\nabla_yU_\theta - s_\theta\|/\|s_\theta\|$ et l'on vérifie l'identité exacte, où
$\nabla_y s_\theta^\top y$ est un produit vecteur-jacobien.

In [ ]:
ts = [1e-8, 1e-6, 1e-4, 1e-2, 1.0, 1e2]
x = X[:8].to(T.DEVICE)
print("%8s %22s %22s %18s" % ("t", "|grad U - s| / |s|", "|grad U - (s+J^T y)/2|", "cos(grad U, s)"))
for t in ts:
    y = (x + math.sqrt(t) * torch.randn_like(x)).requires_grad_(True)
    tt = torch.full((len(y),), t, device=T.DEVICE)
    with torch.enable_grad():
        s = m.unet(y, tt)
        U = 0.5 * (y * s).flatten(1).sum(1)
        gU = torch.autograd.grad(U.sum(), y, retain_graph=True)[0]
        JTy = torch.autograd.grad(s, y, grad_outputs=y)[0]
    n = lambda a: a.flatten(1).norm(dim=1)
    e1 = (n(gU - s) / n(s)).mean().item()
    e2 = (n(gU - 0.5 * (s + JTy)) / n(gU)).mean().item()
    cos = ((gU * s).flatten(1).sum(1) / (n(gU) * n(s))).mean().item()
    print("%8.0e %22.4f %22.2e %18.4f" % (t, e1, e2, cos))

## 3. Homogénéité

Un réseau homogène vérifie $s_\theta(\lambda y, t) = \lambda\, s_\theta(y, t)$, d'où $U_\theta(\lambda y, t)
= \lambda^2 U_\theta(y, t)$ (énergie **sans** la constante de normalisation). On mesure les deux écarts sur des
images propres ($t$ très petit) et bruitées.

In [ ]:
lambdas = [0.25, 0.5, 2.0, 4.0]
for t in [1e-8, 1e-2]:
    y = x + math.sqrt(t) * torch.randn_like(x)
    tt = torch.full((len(y),), t, device=T.DEVICE)
    with torch.no_grad():
        s1 = m.unet(y, tt); U1 = m.energie_brute(y, tt)
        for lam in lambdas:
            sl = m.unet(lam * y, tt); Ul = m.energie_brute(lam * y, tt)
            err_s = ((sl - lam * s1).flatten(1).norm(dim=1) / (lam * s1).flatten(1).norm(dim=1)).mean().item()
            rap = (Ul / (lam ** 2 * U1)).mean().item()
            print("t = %.0e  lambda = %.2f : |s(ly) - l s(y)| / |l s(y)| = %.3f ;  U(ly) / (l^2 U(y)) = %.3f"
                  % (t, lam, err_s, rap))

noire = torch.zeros(1, 1, 64, 64, device=T.DEVICE)
print()
for t in [0.0, 1e-4, 1.0, 1e3]:
    with torch.no_grad():
        u = m.energie_brute(noire, torch.full((1,), t, device=T.DEVICE)).item()
    print("image noire, t = %7.0e : U_theta brute = %.1f nats ; log p normalisée = %.2f dB/dim"
          % (t, u, T.nats_vers_dB(torch.tensor(u + m.constante), d).item()))

## 4. Le réseau utilise-t-il le niveau de bruit ?

Le UNet reçoit $t$ par un encodage de Fourier $(\cos\omega_k t, \sin\omega_k t)$ suivi d'un petit perceptron
(`networks/conditioning.py`). On trace la distance entre l'encodage de $t$ et celui de $t_{\min} = 10^{-9}$, puis on
débruite des images de PSNR d'entrée 40 dB ($t = 10^{-4}$) en donnant au réseau un $t$ faux.

In [ ]:
tgrille = torch.logspace(-9, 3, 200, device=T.DEVICE)
with torch.no_grad():
    e = emb(tgrille); e0 = emb(torch.full((1,), 1e-9, device=T.DEVICE))
dist = (e - e0).norm(dim=1) / e0.norm()
plt.figure(figsize=(5.5, 3.2))
plt.semilogx(tgrille.cpu(), dist.cpu())
plt.xlabel("t"); plt.ylabel("|e(t) - e(1e-9)| / |e(1e-9)|"); plt.title("Encodage du niveau de bruit")
plt.grid(alpha=0.3)
T.montrer("encodage_t")

t_vrai = 1e-4
torch.manual_seed(GRAINE)
xb = X[:16].to(T.DEVICE)
yb = xb + math.sqrt(t_vrai) * torch.randn_like(xb)
print("PSNR d'entrée : %.1f dB" % T.psnr(xb, yb).mean().item())
for t_donne in [1e-9, 1e-6, 1e-4, 1e-2, 1.0]:
    s = m.reseau(m.entree(yb, t_donne), create_graph=False)
    # le débruitage utilise le t donné au réseau : x̂ = y - t_donne * grad U ; on montre aussi x̂ avec le vrai t
    x_donne = yb - t_donne * s.data_score
    x_vrai = yb - t_vrai * s.data_score
    print("t donné %.0e : PSNR de y - t_donné grad U = %6.2f dB ; de y - t_vrai grad U = %6.2f dB"
          % (t_donne, T.psnr(xb, x_donne.detach()).mean().item(), T.psnr(xb, x_vrai.detach()).mean().item()))

## 5. Performances de débruitage (tableau 1)

Les auteurs comparent, en couleur, le réseau d'énergie (score matching dual) et le réseau de score de même
architecture (score matching ordinaire), qui renvoie directement l'image débruitée. Les valeurs de l'article sont
enregistrées dans les points de contrôle (`perf_test`, mesurées pendant l'entraînement sur la validation d'ImageNet64) ;
on les recalcule sur nos images.

In [ ]:
m_coul = T.charger_modele("couleur")
m_score = T.charger_modele("debruiteur_couleur")
psnr_entree = [90, 75, 60, 45, 30, 15, 0, -15, -30]
ref = lambda mm: dict(zip([round(v) for v in (-10 * torch.log10(mm.perf_test.noise_levels.variance)).tolist()],
                          (-10 * torch.log10(mm.perf_test.denoising_errors.mse)).tolist()))
print("PSNR d'entrée              " + " ".join("%7d" % p for p in psnr_entree))
for nom, mm in [("réseau de score", m_score), ("réseau d'énergie", m_coul)]:
    print("%-26s" % (nom + " (auteurs)") + " ".join("%7.2f" % ref(mm)[p] for p in psnr_entree))

Xc = T.charger_imagenet64_val(N_IMAGES, gris=False)
mesures = {}
for nom, mm in [("réseau de score", m_score), ("réseau d'énergie", m_coul)]:
    T.fixer_graine(GRAINE)
    ligne = []
    for p in psnr_entree:
        t = 10 ** (-p / 10)
        Y = Xc + math.sqrt(t) * torch.randn_like(Xc)
        Xh = mm.debruiter(Y, t, taille_lot=32)
        ligne.append((-10 * torch.log10(((Xh - Xc) ** 2).mean())).item())   # MSE moyenne, puis PSNR
    mesures[nom] = ligne
    print("%-26s" % (nom + " (ici)") + " ".join("%7.2f" % v for v in ligne))
gain = np.array(mesures["réseau d'énergie"]) - np.array(mesures["réseau de score"])
print("%-26s" % "écart énergie - score" + " ".join("%7.2f" % v for v in gain))

## 6. Les deux scores sont-ils cohérents ? L'équation de diffusion

L'énergie exacte $U(y,t) = -\log p(y\mid t)$ vérifie (annexe B.3, équation (35))
$\partial_t U = \frac12\Delta_y U - \frac12\|\nabla_y U\|^2$. Le score de temps $\partial_tU_\theta$ et le
score d'espace $\nabla_yU_\theta$ du réseau sont appris par deux termes différents de la perte : cette relation n'est
imposée nulle part. On compare les deux membres, multipliés par $t/d$, le laplacien étant estimé par la méthode de
Hutchinson ($\Delta U = \mathbb E_v[v^\top \nabla^2 U\, v]$, $v$ de Rademacher).

In [ ]:
xh = X[:N_HUTCHINSON].to(T.DEVICE)
print("%8s %16s %22s %12s" % ("t", "t dU/dt / d", "t (lap U - |grad U|^2) / 2d", "écart-type"))
resultats_diffusion = []
for t in [1e-6, 1e-4, 1e-2, 1.0]:
    torch.manual_seed(GRAINE)
    y = xh + math.sqrt(t) * torch.randn_like(xh)
    U, gy, gt, lap, err = T.laplacien_hutchinson(m, y, t, n_sondes=8)
    gauche = (t * gt / d).cpu()
    droite = (t / 2 * (lap - (gy ** 2).flatten(1).sum(1)) / d).cpu()
    incert = (t / 2 * err / d).cpu()
    resultats_diffusion.append((t, gauche, droite, incert))
    print("%8.0e %16.4f %22.4f %12.4f   (moyennes sur %d images ; t lap U / 2d = %.4f, t |grad U|^2 / 2d = %.4f)"
          % (t, gauche.mean(), droite.mean(), incert.mean(), len(y), (t * lap / (2 * d)).mean(),
             (t * (gy ** 2).flatten(1).sum(1) / (2 * d)).mean()))
plt.figure(figsize=(4.5, 4))
for t, g, dr, inc in resultats_diffusion:
    plt.errorbar(g, dr, yerr=inc, fmt="o", ms=4, label="t = %.0e" % t)
lim = [min(min(r[1].min(), r[2].min()) for r in resultats_diffusion), max(max(r[1].max(), r[2].max()) for r in resultats_diffusion)]
plt.plot(lim, lim, "k:", lw=1)
plt.xlabel("t ∂U/∂t / d (score de temps)"); plt.ylabel("t (ΔU − |∇U|²) / 2d (score d'espace)")
plt.legend(fontsize=8); plt.title("Équation de diffusion")
T.montrer("equation_diffusion")

## 7. La constante de normalisation

Équation (9) : à $t_{\max} = 10^3$, $p(y\mid t_{\max}) \approx \mathcal N(m, t_{\max} I)$, d'entropie
$\frac d2\log(2\pi e\, t_{\max})$, d'où $C = \frac d2\log(2\pi e\, t_{\max}) - \mathbb E_y[U_\theta(y, t_{\max})]$.
Le score de temps garantit que la masse totale $\int e^{-U_\theta(y,t)}dy$ ne dépend pas de $t$ : la même constante
doit convenir à tout $t$. On le teste en remplaçant l'entropie gaussienne isotrope par celle de la gaussienne de même
covariance que $y$, $\frac12\sum_i\log\big(2\pi e(\lambda_i + t)\big)$ ($\lambda_i$ : valeurs propres de la
covariance des images), qui majore l'entropie de $p(y\mid t)$ et l'approche quand $t$ est grand.

In [ ]:
Xcov = T.charger_imagenet64_val(10_000)
centre = Xcov.flatten(1) - Xcov.flatten(1).mean(0)
lam = torch.linalg.eigvalsh(centre.T @ centre / len(centre)).clamp_min(0).double()
print("variance totale des images : %.1f ; plus grande valeur propre : %.2f ; variance par pixel %.4f (auteurs %.4f)"
      % (lam.sum(), lam.max(), lam.mean(), m.info.variance))

Xn = T.charger_imagenet64_val(4 * N_IMAGES, debut=N_IMAGES)
print("constante enregistrée par les auteurs : %.1f nats" % m.constante)
for t in [1e3, 1e2, 1e1, 1.0, 1e-1]:
    _, e = T.constante_normalisation(m, Xn, t_max=t)
    h_iso = d / 2 * math.log(2 * math.pi * math.e * t)
    h_gauss = 0.5 * torch.log(2 * math.pi * math.e * (lam + t)).sum().item()
    print("t = %6.0e : E[U_theta brute] = %9.1f ; C (isotrope) = %8.1f ; C (gaussienne de même covariance) = %8.1f"
          % (t, e.mean().item(), h_iso - e.mean().item(), h_gauss - e.mean().item()))